# 06. Baselines: PID+FF and the linear swing damper (arms 1 and 2)

**MRAC-RL logbook, step 2 of 3.** Arm 1 is PID+feedforward with a **mass-blind** feedforward; arm 2 adds a fixed linear swing law (the *softer law* $G(-1,-0.5)$ or the *damper* $G(-1,-2)$) on the torque channel. The damper is derived step by step in `sympy`; the controllers are then run on the offline plant, including a 25-condition grid in the style of the Pymunk sweep S30.

> **EXPLORATORY / offline / PROXY.** Reduced taut point-load plant (`QuadLoadPend`), development PC. Offline metrics are named `c_T_off`, `c_S_off`: they are **not** the canonical `c_T`, `c_S`. No stability claim (the pole analysis is a linear, continuous-time, unsaturated check), no claim about the Pymunk chain plant. Plant theory: MQP (`.ai/research/derivations/mrac-quadrotor-payload.md`) §5; the baseline laws are the package's PID+FF and `LinearSwingResidual`, re-implemented offline in `experiments/mrac/baselines.py` (registry rows EQ-CL-03, -04, -05, EQ-CB-04).

## 1. Research question

1. **Arm 1.** What does it cost the PID+FF baseline to use the nominal payload mass (4 kg) in the feedforward when the real payload is lighter or heavier?
2. **Derivation.** Why does the linear damper add damping to the swing, which signs and scalings make it work, and where does it stop working (linear analysis)?
3. **Arm 2.** Relative to the **true-mass PID+FF** (the denominator of every ratio in this notebook, as in S30) and relative to arm 1, do the softer law and the damper reduce the swing, and what do they cost in tracking, across the S30 grid of cable lengths and payload masses?
4. **Cross-check.** Do the offline trends resemble the Pymunk S30 trends (trends, not values)?

## 2. Hypotheses and predicted direction

Stated before any number below.

| # | Hypothesis | Predicted direction |
|---|---|---|
| H1 | The mass-blind feedforward is the true-mass feedforward when $m_L=4$ kg, so the two arms coincide there (ratio exactly 1). For $m_L>4$ the vehicle sags: a height transient proportional to $(m_L-4)g$. For $m_L<4$ the tension gate $\mathrm{clip}(T/\hat mg,0,1)$ lowers the feedforward to the true weight **at hover only** (the gate reads the instantaneous tension, which moves when the cable swings), so the arms are close but not identical away from hover | $c_{T,off}=1$ at 4 kg; $c_{T,off}>1$ for $m_L>4$, growing with $m_L-4$; close to 1 for $m_L<4$ |
| H2 (informed by lane A, F-4) | In the linear model the damper raises the damping ratio of the least-damped oscillatory closed-loop mode for the two presets, **but not necessarily monotonically in $k_{\dot\phi}$** (the swing mode couples to the translation mode); and the linear time response reproduces the nonlinear episode for small swing | $\zeta_{\min}$ up for $G(-1,-2)$ and $G(-1,-0.5)$ at nominal, with a maximum over $k_{\dot\phi}$ and a decline beyond it; linear/nonlinear mismatch of a few percent of the amplitude |
| H3 | The damper path has a right-half-plane zero for $d>0$ (offset attachment), which guarantees only that *infinite* gain is unstable; a finite instability threshold in $|k_{\dot\phi}|$ may exist in the continuous-time linear model, and its cause is not assumed to be that zero | a finite threshold in $|k_{\dot\phi}|$; compared with the same model at $d=0$ |
| H4 | The fixed form $k_{\dot\phi}\sqrt{L_{nom}/L}$ varies the damping ratio less with $L$ than the oracle form (rate divided by the true $\omega_0$) | smaller spread of $\zeta_{\min}$ over $L$ for the fixed form (a hypothesis that may be refuted) |
| H5 | On the grid, the softer law and the damper reduce swing relative to the true-mass PID+FF ($c_{S,off}<1$) and relative to arm 1 in most conditions, and cost tracking ($c_{T,off}>1$) | $c_{S,off}<1$ and $c_{T,off}>1$; the damper stronger in both |

## 3. Experiment identity

In [1]:
%matplotlib inline
import math, itertools, time, json
from functools import lru_cache
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp
from IPython.display import Markdown, Math, display
from scipy.linalg import expm

from mrac_lab import provenance, style, plots, widgets, grid

prov = provenance.setup(pinned_commit="c16a4f8")   # commit this notebook was written against (Claude)
style.init()

from experiments.mrac.baselines import (
    canonical_offline_params, run_episode, hold_reference, MinJerkReference, s30_conditions,
    S30_LENGTHS, S30_MASSES, S30_L_NOM, S30_M_NOM, SOFTER_LAW_GAINS, DAMPER_GAINS, omega_0,
)
from experiments.mrac.offline_metrics import OffMetrics, episode_metrics, c_T_off, c_S_off

CANON = canonical_offline_params()
VEH, G = CANON.quad, CANON.quad.g
SOURCES = {r.name + " [" + r.unit + "]": "%s  <- %s" % (r.value, r.source) for r in CANON.records}
provenance.banner(prov, "06 Baselines: PID+FF and the linear damper", SOURCES,
                  notes=list(CANON.caveats) + [
                      "Controllers: true-mass PID+FF (the denominator of every ratio); arm 1 = PID+FF with mass-blind FF (m_hat = 4 kg, S30 PIDFF-nom); "
                      "arm 2 = softer law G(-1,-0.5) with mass-blind FF; damper G(-1,-2) with mass-blind FF; s30match = softer law with TRUE-mass FF. "
                      "G(...) = FIXED form kd sqrt(L_nom/L) (S30 convention).",
                      "FF tension gate: the offline gate uses the instantaneous tension of the current state (ideal, unfiltered), unlike S30's median-filtered tension.",
                      "Scenarios: " + "; ".join(f"{k}: {v['text']}" for k, v in grid.SCENARIOS.items())])

06 Baselines: PID+FF and the linear damper
Label: EXPLORATORY / offline / PROXY (development PC, reduced taut point-load model)
Thesis checkout: C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load
Commit: c16a4f8986263d7323cf6299df1b10a5fce8b54e (branch Claude, WORKING TREE HAS UNCOMMITTED CHANGES under experiments/ or gym_pymunk_quadrotor/)
Python: 3.12.3
Claim limits: No stability claim, no Pymunk-transfer claim, no hardware claim. Offline metrics are named *_off and are not the canonical c_T / c_S.
import origin: experiments.mrac.quad -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load\experiments\mrac\quad.py
import origin: experiments.mrac.quad_load_pend -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load\experiments\mrac\quad_load_pend.py
import origin: experiments.notebook_utils.style ->

m_q [kg],9.6 <- gym_pymunk_quadrotor/plant/config.py:61
width [m],1.164 <- gym_pymunk_quadrotor/plant/config.py:62
height [m],0.759 <- gym_pymunk_quadrotor/plant/config.py:63
J [kg m^2],1.5447816 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:94-95 (m (w^2+h^2)/12)
a_p = a_n [m],"0.582 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:96-97,301 (width/2)"
vehicle max_thrust (total) [N],"201.036 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:157-159 ((m+max_payload) g + m max_ascent_speed; config.py:66,69)"
T_max (per motor) [N],100.518 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:300; plant/entities/motor.py:135 (thrust = max_thrust_per_motor cmd^2)
tau_m [s],0.05 <- gym_pymunk_quadrotor/plant/config.py:72; motor.py:92-93 (first-order lag)
g [m/s^2],9.81 <- gym_pymunk_quadrotor/config/physics_legacy.py:25
kappa_L = kappa_R [-],1.0 <- no effectiveness split in Pymunk
d [m],"0.3795 <- gym_pymunk_quadrotor/plant/xpbd_cable.py:330-333 (pivot_local = (0, -height/2): cable leaves the bottom edge)"


## 4. Theory

### 4.1 Arm 1: PID + feedforward (registry EQ-CL-05, EQ-CL-03)

Errors are target minus measured. With reference $(y_r,z_r,\dot y_r,\dot z_r,\ddot y_r,\ddot z_r)$ and the **vehicle** mass $m$ (the PID never sees the payload):
$$
a_{d,y}=k_{p,y}e_y+k_{d,y}\dot e_y+k_{i,y}\!\int\! e_y+\ddot y_r,\qquad
a_{d,z}=k_{p,z}e_z+k_{d,z}\dot e_z+k_{i,z}\!\int\! e_z+g+\ddot z_r,
$$
$$
u_{\rm PID}=\frac{m\,a_{d,z}}{\cos\theta},\qquad
\theta_d=\arcsin\!\Big(\mathrm{clip}\Big(\frac{-m\,a_{d,y}}{u_{\rm PID}}\Big)\Big),\qquad
\tau_{\rm PID}=-J\big[k_{p,\theta}(\theta-\theta_d)+k_{d,\theta}(\omega-\dot\theta_d)\big],
$$
with back-calculation anti-windup and clips (commanded tilt, thrust, torque). The **feedforward** adds the payload weight with a *nominal* mass $\hat m$, gated by the measured tension:
$$
f_{\rm FF}=\frac{\hat m\,g}{\max(\cos\theta,\cos\theta_{\lim})}\;\mathrm{clip}\!\Big(\frac{T}{\hat m g},0,1\Big),\qquad f_{\rm req}=u_{\rm PID}+f_{\rm FF},\quad \tau_{\rm req}=\tau_{\rm PID}.
$$
*True-mass* means $\hat m=m_L$; *mass-blind* means $\hat m=4$ kg for every episode (S30 "PIDFF-nom"). **Gate wording:** in this offline model the gate reads the *instantaneous* tension of the current state (ideal, unfiltered, through the previous tick's end tension); the Pymunk S30 runs use a median-filtered tension. Hover needs $(m+m_L)g$ in total, so at hover $T=m_Lg$, so the gate makes $f_{\rm FF}=\min(\hat m,m_L)\,g$. A mass-blind $\hat m$ therefore leaves a static deficit $(m_L-\hat m)g$ only when the payload is **heavier** than assumed (the vertical integrator must absorb it: the vehicle sags); for a lighter payload the gate removes the surplus and the feedforward equals the true weight.

### 4.2 Arm 2: the linear swing law (registry EQ-CL-01, EQ-CL-04, EQ-CB-04)

The law adds a torque, nothing on the collective:
$$
u=u_{\rm PID+FF}+\beta\odot(W\odot a),\quad W=[F_{\max},\tau_{\max}],\ a=[0,a_\tau],\ \beta_\tau=0.3,
$$
$$
a_\tau=\mathrm{clip}\Big(k_\phi\,\phi_{rel}+k_{\dot\phi}\,\frac{\dot\phi_{rel}}{\omega_0},\,-1,\,1\Big),\quad \phi_{rel}=\phi-\phi_{eq},\quad \omega_0=\sqrt{g/L},
$$
with $\phi_{eq}=-\arctan\!\big(a_{y,r}/(g+a_{z,r})\big)$ from the commanded reference acceleration (notebook 05, section 5.6; floor $0.3g$ on the denominator). Whether $\phi_{eq}$ is evaluated at the same instant as in the Pymunk environment is **NOT VERIFIED**. Softer law $G(k_\phi,k_{\dot\phi})=G(-1,-0.5)$, damper $G(-1,-2)$. The **oracle form** divides the rate by the true $\omega_0$; the **fixed form** uses $k_{\dot\phi}\sqrt{L_{nom}/L}$, i.e. divides by $\omega_0(L_{nom})$: a fixed physical gain. Following S30, the label $G(k_\phi,k_{\dot\phi})$ is used **only for the fixed form**; oracle runs are labelled "oracle form". The physical torque gains are
$$
K_\phi=\beta_\tau\tau_{\max}k_\phi,\qquad K_{\dot\phi}=\beta_\tau\tau_{\max}\,k_{\dot\phi}/\omega_0\quad(\omega_0=\omega_0(L)\text{ oracle, }\omega_0(L_{nom})\text{ fixed}).
$$
Sign check (z-up, $b=[-\sin\theta,\cos\theta]$): payload at $\phi>0$ (towards $+y$) needs the vehicle to move towards $+y$, i.e. thrust towards $+y$, i.e. $\theta<0$, i.e. $\tau<0$: hence $k_\phi<0$.

### 4.3 Derivation of the damper's effect (linear, continuous time)

**Assumptions** (the regime in which every statement of this subsection holds):

| | Assumption |
|---|---|
| A1 | Small angles: $\theta,\phi$ and their rates are first order; products of deviations are dropped. |
| A2 | Taut cable ($T=m_Lg>0$ at hover), point payload, rigid massless cable (MQP §5). |
| A3 | No saturation anywhere: no clip in the PID, the damper ($|a_\tau|<1$), the allocator. |
| A4 | The vertical loop is decoupled: $b_z=\cos\theta=1+O(\theta^2)$, so $f=f_0+\delta f$ only affects $z$ at first order. |
| A5 | Continuous time: the 50 Hz ZOH (about half a sample, 10 ms, of delay) and the finite-difference tilt-rate feedforward are replaced by their continuous counterparts. Motor lag is kept as a first-order lag $1/(\tau_ms+1)$ on the torque (an option below). |
| A6 | Reference at rest at hover ($\phi_{eq}=0$, no reference acceleration): this is *regulation*; the moving $\phi_{eq}$ only adds a forcing term. |
| A7 | The vertical integrator has settled, so $u_{\rm PID}=(m+m_L-\min(\hat m,m_L))g$ and the tilt map is $\theta_d=-m\,a_{d,y}/u_{\rm PID}=-a_{d,y}/g_e$ with $g_e=g(1+\varepsilon_{ff})$, $\varepsilon_{ff}=\max(m_L-\hat m,0)/m$ ($\varepsilon_{ff}=0$ for the true-mass FF and for $m_L\le\hat m$). |
| A8 | Hover thrust $f_0=(m+m_L)g$, constant parameters; $d$ is the (known) attachment offset. |

Every step has a `sympy` check; the cells assert the identities (they would raise if a step were wrong).

In [2]:
# ---- Step 1: linearise the nonlinear model at hover (symbolic, from MQP 5.6 and 5.5) ------------------
def show(label, expr=None):
    display(Math(label if expr is None else label + sp.latex(expr)))

t_, eps = sp.symbols("t epsilon")
m_, J_, mL_, l_, d_, g_, f0 = sp.symbols("m J m_L l d g f_0", positive=True)
y_, z_, th_, ph_, tau_, df_ = [sp.Function(n)(t_) for n in ("y", "z", "theta", "phi", "tau", "delta_f")]
sig, gam = sp.sin(ph_ - th_), sp.cos(ph_ - th_)
w_vec = sp.Matrix([sp.sin(ph_), -sp.cos(ph_), d_ * sig])                          # w = [q_y, q_z, d sigma]
M_mat = sp.diag(m_, m_, J_) + mL_ * w_vec * w_vec.T                                # MQP 5.6
f_ = sp.Symbol("f")
U_vec = sp.Matrix([-f_ * sp.sin(th_), f_ * sp.cos(th_), tau_])                    # U = [f b(theta), tau]
Tbar0 = g_ * sp.cos(ph_) + l_ * ph_.diff(t_) ** 2 + d_ * gam * th_.diff(t_) ** 2
xi_dd = sp.Matrix([y_.diff(t_, 2), z_.diff(t_, 2), th_.diff(t_, 2)])
nonlinear = M_mat * xi_dd - (U_vec - m_ * g_ * sp.Matrix([0, 1, 0]) + mL_ * Tbar0 * w_vec)   # = 0
swing_nl = (l_ * ph_.diff(t_, 2) + g_ * sp.sin(ph_)
            + (sp.Matrix([sp.cos(ph_), sp.sin(ph_)]).T * sp.Matrix([y_.diff(t_, 2), z_.diff(t_, 2)]))[0]
            + d_ * gam * th_.diff(t_, 2) + d_ * sig * th_.diff(t_) ** 2)                      # MQP 5.5 = 0

def linearise(expr):
    e = expr.subs({f_: f0 + eps * df_}).subs(
        {y_: eps * y_, z_: eps * z_, th_: eps * th_, ph_: eps * ph_, tau_: eps * tau_}, simultaneous=True).doit()
    e = sp.series(e, eps, 0, 2).removeO()
    return sp.simplify(e.coeff(eps, 1).subs(f0, (m_ + mL_) * g_)), sp.simplify(e.coeff(eps, 0).subs(f0, (m_ + mL_) * g_))

lin = [linearise(nonlinear[i]) for i in range(3)] + [linearise(swing_nl)]
by_hand = [m_ * y_.diff(t_, 2) + (m_ + mL_) * g_ * th_ - mL_ * g_ * ph_,        # lateral
           (m_ + mL_) * z_.diff(t_, 2) - df_,                                    # vertical
           J_ * th_.diff(t_, 2) - tau_ - mL_ * g_ * d_ * (ph_ - th_),            # attitude (cable torque d T sigma)
           l_ * ph_.diff(t_, 2) + g_ * ph_ + y_.diff(t_, 2) + d_ * th_.diff(t_, 2)]   # swing
for (first, zeroth), hand in zip(lin, by_hand):
    assert sp.simplify(zeroth) == 0, "hover is not an equilibrium"
    assert sp.simplify(first - hand) == 0, "linearisation differs from the hand-derived equation"
show(r"\text{Hover is an equilibrium of all four equations (order }\epsilon^0\text{ vanishes).}")
show(r"m\ddot y+(m+m_L)g\,\theta-m_Lg\,\phi=0 \quad\text{(lateral)}")
show(r"J\ddot\theta-\tau-m_Lgd\,(\phi-\theta)=0 \quad\text{(attitude; cable torque } dT\sigma\text{ with }T=m_Lg)")
show(r"l\ddot\phi+g\phi+\ddot y+d\,\ddot\theta=0 \quad\text{(swing)}")
show(r"(m+m_L)\ddot z=\delta f \quad\text{(vertical, decoupled: A4)}")
print("sympy: first-order terms equal the hand-derived equations; zeroth-order terms vanish at f_0 = (m+m_L) g.")
# sanity limit (MQP 5.7): d = 0, theta = 0, free vehicle -> omega^2 = (g/l)(1 + m_L/m)
om2 = sp.symbols("omega2", positive=True)
lat_free = (m_ * y_.diff(t_, 2) - mL_ * g_ * ph_).subs(th_, 0)             # lateral equation, theta = 0
swing_free = (l_ * ph_.diff(t_, 2) + g_ * ph_ + y_.diff(t_, 2))              # swing equation, d = 0
yddot = sp.solve(lat_free, y_.diff(t_, 2))[0]
harmonic = sp.simplify(swing_free.subs(y_.diff(t_, 2), yddot).subs(ph_.diff(t_, 2), -om2 * ph_) / ph_)
print("limit check d=0, free vehicle: omega^2 =", sp.solve(harmonic, om2)[0], " (expected (g/l)(1+m_L/m))")
assert sp.simplify(sp.solve(harmonic, om2)[0] - g_ / l_ * (1 + mL_ / m_)) == 0

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

sympy: first-order terms equal the hand-derived equations; zeroth-order terms vanish at f_0 = (m+m_L) g.
limit check d=0, free vehicle: omega^2 = g*(m + m_L)/(l*m)  (expected (g/l)(1+m_L/m))


**Step 2: the controller, linearised.** With the reference at rest at the origin, $e_y=-y$. Under A7 the tilt command is $\theta_d=\big(k_{p,y}y+k_{d,y}\dot y+k_{i,y}\!\int y\big)/g_e$ (a displaced vehicle tilts *towards* the reference, $\theta_d>0$ for $y>0$), and the attitude loop is $\tau_{\rm PID}=-J[k_{p,\theta}(\theta-\theta_d)+k_{d,\theta}(\dot\theta-\dot\theta_d)]$. The damper adds $\tau_d=K_\phi\phi+K_{\dot\phi}\dot\phi$. The torque reaching the vehicle is the lagged command, $\tau=(\tau_{\rm PID}+\tau_d)/(\tau_m s+1)$.

**Step 3: characteristic polynomial.** Laplace transform of the four equations with the unknowns $(Y,\Theta,\Phi)$; the closed-loop polynomial is the determinant of the $3\times3$ system (times $s$ to clear the integrator's $1/s$). Because the damper enters **only** the attitude equation and **only** through $\Phi$, the polynomial is *affine* in the damper gains:
$$
P(s)=P_0(s)+\big(K_\phi+K_{\dot\phi}\,s\big)\,N(s),
$$
where $P_0$ is the arm-1 polynomial and $N(s)$ the cofactor of that entry, which does not depend on any controller gain.

In [3]:
# ---- Steps 2-3: closed-loop polynomial in the Laplace domain ----------------------------------------------
s = sp.symbols("s")
ge = sp.symbols("g_e", positive=True)
kpy, kdy, kiy, kpt, kdt, Kphi, Kphid, taum = sp.symbols("k_py k_dy k_iy k_ptheta k_dtheta K_phi K_phidot tau_m", positive=False)
Ys, Ths, Phs = sp.symbols("Y Theta Phi")

theta_d = (kpy + kdy * s + kiy / s) * Ys / ge                                    # step 2: tilt command
tau_cmd = -J_ * (kpt + kdt * s) * (Ths - theta_d) + (Kphi + Kphid * s) * Phs        # PID attitude loop + damper
eq_lat = m_ * s**2 * Ys + g_ * (m_ + mL_) * Ths - g_ * mL_ * Phs
eq_swing = l_ * s**2 * Phs + g_ * Phs + s**2 * Ys + d_ * s**2 * Ths
def system(lag):
    eq_att = (1 + taum * s) * (J_ * s**2 * Ths - d_ * g_ * mL_ * (Phs - Ths)) - tau_cmd if lag else \
             J_ * s**2 * Ths - d_ * g_ * mL_ * (Phs - Ths) - tau_cmd
    A, _ = sp.linear_eq_to_matrix([eq_lat, eq_att, eq_swing], [Ys, Ths, Phs])
    return sp.expand(sp.together(A.det() * s).as_numer_denom()[0])

P_nolag, P_lag = system(False), system(True)
parts = {}
for name, P in (("no motor lag", P_nolag), ("with motor lag", P_lag)):
    P0 = P.subs({Kphi: 0, Kphid: 0})
    N = sp.factor(sp.simplify((P - P0) / (Kphi + Kphid * s)))
    assert not ({Kphi, Kphid, kpy, kdy, kiy, kpt, kdt, taum} & N.free_symbols), "N must not depend on any gain"
    parts[name] = (P0, N, sp.Poly(P, s).degree())
(P0_nl, N_nl, deg_nl), (P0_l, N_l, deg_l) = parts["no motor lag"], parts["with motor lag"]
assert sp.simplify(N_nl - N_l) == 0, "the damper numerator is unchanged by the motor lag"
print("degree of P(s): %d without motor lag, %d with motor lag (integrator included)" % (deg_nl, deg_l))
show(r"N(s)=", N_l)
z_rhp = sp.sqrt(g_ * (m_ + mL_) / (d_ * m_))
show(r"\text{finite zeros of the damper path: } s=\pm", z_rhp)
print("numerically (canonical): +-%.3f rad/s (right-half-plane zero for d > 0); for d = 0 there is no finite zero." %
      float(z_rhp.subs({g_: G, m_: VEH.m, mL_: CANON.payload.m_L, d_: CANON.payload.d})))
print("dependence of the zero on m_L and d (rad/s):")
for _mL in (2.0, 4.0, 6.0):
    print("  m_L = %g kg: " % _mL + ", ".join("d = %.4f m: %.2f" % (_d, float(z_rhp.subs({g_: G, m_: VEH.m, mL_: _mL, d_: _d}))) for _d in (0.1, CANON.payload.d, 0.6)))
print("reduced point-attachment model only; no claim for the Pymunk chain.")

degree of P(s): 7 without motor lag, 8 with motor lag (integrator included)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

numerically (canonical): +-6.051 rad/s (right-half-plane zero for d > 0); for d = 0 there is no finite zero.
dependence of the zero on m_L and d (rad/s):
  m_L = 2 kg: d = 0.1000 m: 10.89, d = 0.3795 m: 5.59, d = 0.6000 m: 4.44
  m_L = 4 kg: d = 0.1000 m: 11.79, d = 0.3795 m: 6.05, d = 0.6000 m: 4.81
  m_L = 6 kg: d = 0.1000 m: 12.63, d = 0.3795 m: 6.48, d = 0.6000 m: 5.15
reduced point-attachment model only; no claim for the Pymunk chain.


**Reading $N(s)$.** $N(s)=g_e\,s^{3}\,\big(d\,m\,s^{2}-g(m+m_L)\big)$: three zeros at the origin and a symmetric real pair $s=\pm\sqrt{g(m+m_L)/(d\,m)}$. The **right-half-plane zero exists only for $d>0$** and is independent of every controller gain. Its cause is the **offset attachment**, not the cable torque $d\,T\sigma$: through the $d\,\ddot\theta$ term of the swing equation the attachment point first moves opposite to where the tilt later drives it. Its value depends on the payload mass and on the offset: $\sqrt{g(m+m_L)/(d\,m)}$ is about 6.05 rad/s at $m_L=4$ kg and $d=0.3795$ m (the cell above prints it for $m_L=2,4,6$ kg and other $d$); it belongs to this reduced point-attachment model and does **not** transfer to the Pymunk chain. Consequence for the root locus $1+(K_\phi+K_{\dot\phi}s)N(s)/P_0(s)=0$: as the gain tends to infinity the branches end on the zeros of $N$, including $+\sqrt{g(m+m_L)/(dm)}$, so the infinite-gain limit is necessarily unstable. That is all the zero guarantees. In the finite sweep shown below the first instability to appear is a lightly damped fast complex pair reaching the imaginary axis (right panel), a crossing that is **not** attributed to the zero (section 6 repeats the sweep at $d=0$); the threshold is read numerically (H3) in a continuous-time, unsaturated linear model. Whether a branch reaches the right-half-plane zero at still higher gain was not examined.

**Step 4: numbers, and an independent state-space check.** The same equations are written as a state-space model $\dot x=Ax$ with $x=(y,\dot y,\theta,\dot\theta,\phi,\dot\phi,\!\int\!y,\tau_{act})$ and the eigenvalues compared with the roots of $P(s)$.

In [4]:
# ---- Step 4: numbers; state-space form of the same equations; cross-check with the roots of P(s) ------------
GN = CANON.gains
def physical_damper_gains(kp, kd, L, fixed):
    w0 = math.sqrt(G / (CANON.L_nom if fixed else L))
    return CANON.beta_tau * CANON.tau_max * kp, CANON.beta_tau * CANON.tau_max * kd / w0

def closed_loop_A(kp, kd, L=None, m_L=None, fixed=True, lag=True, eps_ff=0.0, integ=True, d=None):
    # state x = (y, y', theta, theta', phi, phi', int y, tau_act); lag=False replaces tau_act by tau_cmd (7 states used)
    m, J = VEH.m, VEH.J
    d = CANON.payload.d if d is None else d
    L = CANON.payload.l if L is None else L
    m_L = CANON.payload.m_L if m_L is None else m_L
    g_e = G * (1.0 + eps_ff)
    Kp_, Kd_ = physical_damper_gains(kp, kd, L, fixed)
    I = np.eye(8)
    y_acc = (-(m + m_L) * G * I[2] + m_L * G * I[4]) / m
    kiy_ = GN.ki_y if integ else 0.0
    th_d = (GN.kp_y * I[0] + GN.kd_y * I[1] + kiy_ * I[6]) / g_e
    dth_d = (GN.kp_y * I[1] + kiy_ * I[0]) / g_e + GN.kd_y / g_e * y_acc
    tau_cmd = -J * (GN.kp_theta * (I[2] - th_d) + GN.kd_theta * (I[3] - dth_d)) + Kp_ * I[4] + Kd_ * I[5]
    tau = I[7] if lag else tau_cmd
    th_acc = (tau + m_L * G * d * (I[4] - I[2])) / J
    ph_acc = (-G * I[4] - y_acc - d * th_acc) / L
    A = np.zeros((8, 8))
    A[0] = I[1]; A[1] = y_acc; A[2] = I[3]; A[3] = th_acc; A[4] = I[5]; A[5] = ph_acc; A[6] = I[0]
    A[7] = (tau_cmd - I[7]) / VEH.tau_m if lag else -I[7]
    if not lag:                      # remove the decoupled placeholder state
        A = A[:7, :7]
    return A

def poles(kp, kd, **kw):
    return np.linalg.eigvals(closed_loop_A(kp, kd, **kw))

def track_swing(kp, kds, **kw):
    # follow the swing mode from kd = 0 by nearest-neighbour continuation; returns (zeta, pole) arrays (mode identity is NOT
    # the same as "least damped": the least-damped mode switches along the sweep)
    L = kw.get("L") or CANON.payload.l; m_L = kw.get("m_L") or CANON.payload.m_L
    ev = poles(kp, 0.0, **kw); cx = ev[ev.imag > 1e-9]
    target = math.sqrt(G / L * (1 + m_L / VEH.m))
    cur = min(cx, key=lambda p: abs(p.imag - target)); zs, ps = [], []
    for k in kds:
        ev = poles(kp, k, **kw); cx = ev[ev.imag > 1e-9]
        cur = min(cx, key=lambda p: abs(p - cur)); zs.append(-cur.real / abs(cur)); ps.append(cur)
    return np.array(zs), np.array(ps)

def zeta_summary(ev):
    cx = ev[ev.imag > 1e-9]
    if cx.size == 0:
        return math.nan, float(np.max(ev.real))
    return float(np.min(-cx.real / np.abs(cx))), float(np.max(ev.real))

num = {m_: VEH.m, J_: VEH.J, mL_: CANON.payload.m_L, l_: CANON.payload.l, d_: CANON.payload.d, g_: G, ge: G,
       kpy: GN.kp_y, kdy: GN.kd_y, kiy: GN.ki_y, kpt: GN.kp_theta, kdt: GN.kd_theta, taum: VEH.tau_m}
rows, worst = [], 0.0
for name, (kp, kd) in (("arm 1 (no damper)", (0.0, 0.0)), ("softer law G(-1,-0.5)", SOFTER_LAW_GAINS), ("damper G(-1,-2)", DAMPER_GAINS)):
    Kp_, Kd_ = physical_damper_gains(kp, kd, CANON.payload.l, fixed=True)
    for lag, P in ((True, P_lag), (False, P_nolag)):
        roots_P = np.roots([float(c.subs({**num, Kphi: Kp_, Kphid: Kd_})) for c in sp.Poly(P, s).all_coeffs()])
        ev = poles(kp, kd, lag=lag)
        worst = max(worst, float(np.max(np.abs(np.sort_complex(roots_P) - np.sort_complex(ev)))))
        zmin, rmax = zeta_summary(ev)
        rows.append(dict(controller=name, motor_lag=lag, zeta_min=zmin, max_real_part=rmax,
                         oscillatory_modes=", ".join("%.2f%+.2fj" % (p.real, abs(p.imag)) for p in sorted(ev[ev.imag > 1e-9], key=abs))))
assert worst < 1e-6, worst
print("max |eig(A) - roots(P)| over the 6 cases: %.2e  (state-space form and polynomial agree)" % worst)
STEP4 = pd.DataFrame(rows)
display(STEP4.style.format({"zeta_min": "{:.3f}", "max_real_part": "{:.3f}"}).hide(axis="index"))

max |eig(A) - roots(P)| over the 6 cases: 3.70e-14  (state-space form and polynomial agree)


controller,motor_lag,zeta_min,max_real_part,oscillatory_modes
arm 1 (no damper),True,0.158,-0.317,"-1.02+0.79j, -0.32+1.98j, -7.20+16.44j"
arm 1 (no damper),False,0.170,-0.338,"-1.02+0.77j, -0.34+1.96j"
"softer law G(-1,-0.5)",True,0.244,-0.356,"-0.56+0.86j, -0.60+2.41j, -7.21+16.10j"
"softer law G(-1,-0.5)",False,0.265,-0.356,"-0.57+0.86j, -0.66+2.39j"
"damper G(-1,-2)",True,0.378,-0.361,"-0.44+1.07j, -1.68+2.81j, -6.96+14.95j"
"damper G(-1,-2)",False,0.375,-0.361,"-0.43+1.08j, -1.87+2.94j"


**Step 5: damping ratio against the gains (interactive).** Poles of the closed loop (all 8 with the motor lag) for the chosen law. Left: pole map with the arm-1 poles (grey) and the damped-ratio rays. Middle: two damping ratios against the rate gain $k_{\dot\phi}$ at the chosen $k_\phi$: $\zeta_{\min}$ of the *least-damped* oscillatory mode (blue; **which mode that is changes along the sweep**, so this curve can have kinks) and $\zeta$ of the **swing mode tracked by continuation** from $k_{\dot\phi}=0$ (green dashed); orange: the largest real part (stable if negative); the vertical dashed line is the selected gain. Neither curve is assumed monotone. Right: the locus of all poles as $|k_{\dot\phi}|$ grows from 0, showing the fast pair that approaches the imaginary axis (H3); the stars mark the zeros of $N(s)$.

In [5]:
def draw_poles(kp=-1.0, kd=-2.0, l=2.0, m_L=4.0, form="fixed", lag=True, eps_ff=0.0):
    fixed = form == "fixed"
    kw = dict(L=l, m_L=m_L, fixed=fixed, lag=lag, eps_ff=eps_ff)
    ev0, ev1 = poles(0.0, 0.0, **kw), poles(kp, kd, **kw)
    kds = np.linspace(0.0, -10.0, 201)
    sweep = [zeta_summary(poles(kp, k, **kw)) for k in kds]
    zs = np.array([a for a, b in sweep]); rs = np.array([b for a, b in sweep])
    zsw, _ = track_swing(kp, kds, **kw)
    unstable = np.flatnonzero(rs > 0)
    kd_limit = kds[unstable[0]] if unstable.size else None
    fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.8))
    for a_ in ax: a_.grid(True, alpha=0.3)
    ax[0].plot(ev0.real, ev0.imag, "x", color="#8A8A8A", ms=8, label="no damper (arm 1)")
    ax[0].plot(ev1.real, ev1.imag, "o", mfc="none", color=style.arm_color("damper"), ms=8, label=f"G({kp:g},{kd:g}), {form} form")
    for zeta in (0.1, 0.3, 0.5, 0.7):
        r = 5.0; ang = math.acos(zeta); ax[0].plot([0, -r * math.cos(ang)], [0, r * math.sin(ang)], ":", color="0.8"); ax[0].plot([0, -r * math.cos(ang)], [0, -r * math.sin(ang)], ":", color="0.8")
    ax[0].axvline(0, color=style.SLACK_COLOR, lw=0.8); ax[0].set_xlim(-8, 1); ax[0].set_ylim(-5, 5)
    ax[0].set_xlabel("Re s [1/s]"); ax[0].set_ylabel("Im s [rad/s]"); ax[0].set_title(r"closed-loop poles (rays: $\zeta$ = 0.1, 0.3, 0.5, 0.7)", fontsize=8); ax[0].legend(fontsize=7, loc="lower left")
    ax[1].plot(-kds, zs, color="#0072B2", label=r"$\zeta_{\min}$ (least damped mode; identity switches)")
    ax[1].plot(-kds, zsw, color="#009E73", ls="--", label=r"$\zeta$ of the tracked swing mode")
    ax[1].set_xlabel(r"$|k_{\dot\phi}|$"); ax[1].set_ylabel(r"$\zeta$"); ax[1].axvline(-kd, color="k", ls="--", lw=0.8)
    ax[1].axhline(zeta_summary(ev0)[0], **style.arm_style("ref", label="no damper, least damped"))
    ax2 = ax[1].twinx(); ax2.plot(-kds, rs, color="#D55E00", lw=1.0, label="max Re(pole)"); ax2.axhline(0, color=style.SLACK_COLOR, lw=0.6); ax2.set_ylabel("max Re(pole) [1/s]", fontsize=7)
    ax[1].set_title("damping and stability vs rate gain" + (" (unstable from |kd|=%.1f)" % -kd_limit if kd_limit is not None else " (stable over the sweep)"), fontsize=8)
    ax[1].legend(fontsize=7, loc="upper left")
    sc = ax[2].scatter(np.concatenate([poles(kp, k, **kw).real for k in kds[::5]]), np.concatenate([poles(kp, k, **kw).imag for k in kds[::5]]),
                       c=np.repeat(-kds[::5], 8 if lag else 7), cmap="viridis", s=8)
    z_ = math.sqrt(G * (VEH.m + m_L) / (CANON.payload.d * VEH.m))
    ax[2].plot([z_, -z_], [0, 0], "k*", ms=9, label=f"zeros of N(s): +-{z_:.2f}")
    ax[2].axvline(0, color=style.SLACK_COLOR, lw=0.8); ax[2].set_xlim(-14, 8); ax[2].set_ylim(-20, 20)
    ax[2].set_xlabel("Re s"); ax[2].set_title(r"all poles as $|k_{\dot\phi}|$ grows (colour)", fontsize=8); ax[2].legend(fontsize=7)
    fig.colorbar(sc, ax=ax[2], fraction=0.046, pad=0.02).set_label(r"$|k_{\dot\phi}|$", fontsize=7)
    fig.tight_layout(); return fig

panel_poles = widgets.param_panel({
    "kp": ("slider", -3.0, 0.0, 0.1, -1.0, "k_phi (kp)"),
    "kd": ("slider", -10.0, 0.0, 0.1, -2.0, "k_phidot (kd)"),
    "l": ("slider", 1.0, 3.0, 0.25, 2.0, "cable l [m]"),
    "m_L": ("slider", 2.0, 6.0, 1.0, 4.0, "payload m_L [kg]"),
    "form": ("dropdown", ["fixed", "oracle"], "fixed", "damper form"),
    "lag": ("check", True, "motor lag"),
    "eps_ff": ("slider", 0.0, 0.3, 0.05, 0.0, "eps_ff (A7, heavy load)"),
}, draw_poles)

**Step 6: why $\dot\phi/\omega_0$, and what the fixed form does.** In the dimensionless time $\sigma=\omega_0t$ with $\omega_0=\sqrt{g/l}$ the swing equation reads $\phi''+\phi+(\ddot y+d\ddot\theta)/g=0$: the pendulum part contains no $l$ at all, and the law's argument $k_\phi\phi+k_{\dot\phi}\,\phi'$ is a function of dimensionless states with dimensionless gains. That is the **purpose** of the $\omega_0$ normalisation (oracle form: the gain *scales with the pendulum*). Two caveats follow from the algebra above, and are tested numerically below rather than assumed:

* The torque gain in physical units is $K_{\dot\phi}=\beta_\tau\tau_{\max}k_{\dot\phi}/\omega_0$. The oracle form makes it grow like $\sqrt{l}$; the fixed form keeps it constant. Which keeps the *damping ratio* of the closed loop constant depends on how fast the vehicle side is (attitude loop $\approx$ 5-19 rad/s, motor lag 20 rad/s, translation): the dimensionless pendulum is scale-free only if those are fast against $\omega_0$ (time-scale separation, not exact).
* The fixed form $k_{\dot\phi}\sqrt{L_{nom}/L}$ is *defined* as dividing by $\omega_0(L_{nom})$: at $L=L_{nom}$ both forms coincide (S30 V3).

In [6]:
def draw_scale(kp=-1.0, kd=-2.0, m_L=4.0, lag=True):
    Ls = np.linspace(1.0, 3.0, 17)
    z = {k: [] for k in ("none", "oracle", "fixed")}
    for L in Ls:
        z["none"].append(zeta_summary(poles(0.0, 0.0, L=L, m_L=m_L, lag=lag))[0])
        z["oracle"].append(zeta_summary(poles(kp, kd, L=L, m_L=m_L, fixed=False, lag=lag))[0])
        z["fixed"].append(zeta_summary(poles(kp, kd, L=L, m_L=m_L, fixed=True, lag=lag))[0])
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    ax[0].plot(Ls, z["none"], **style.arm_style("pid_ff", label="no damper")); ax[0].plot(Ls, z["oracle"], color="#CC79A7", label="oracle form"); ax[0].plot(Ls, z["fixed"], **style.arm_style("damper", label="fixed form"))
    ax[0].set_xlabel("cable length l [m]"); ax[0].set_ylabel(r"$\zeta_{\min}$"); ax[0].set_title("least-damped mode vs cable length", fontsize=8); ax[0].legend(fontsize=7)
    ax[1].plot(Ls, [physical_damper_gains(kp, kd, L, False)[1] for L in Ls], color="#CC79A7", label="oracle: grows as sqrt(l)")
    ax[1].plot(Ls, [physical_damper_gains(kp, kd, L, True)[1] for L in Ls], **style.arm_style("damper", label="fixed: constant"))
    ax[1].set_xlabel("l [m]"); ax[1].set_ylabel(r"$K_{\dot\phi}$ [N m s/rad]"); ax[1].set_title("physical rate gain", fontsize=8); ax[1].legend(fontsize=7)
    wfree = []
    for L in Ls:
        target = math.sqrt(G / L * (1 + m_L / VEH.m)); cx = [p for p in poles(0.0, 0.0, L=L, m_L=m_L, lag=lag) if p.imag > 1e-9]
        wfree.append(min(cx, key=lambda p: abs(p.imag - target)).imag)
    ax[2].plot(Ls, wfree, color="#0072B2", label="closed-loop mode nearest the free swing (no damper)"); ax[2].plot(Ls, [math.sqrt(G / L) for L in Ls], **style.arm_style("ref", label=r"$\sqrt{g/l}$"))
    ax[2].plot(Ls, [math.sqrt(G / L * (1 + m_L / VEH.m)) for L in Ls], color="#999999", ls="--", label=r"$\sqrt{(g/l)(1+m_L/m)}$")
    ax[2].set_xlabel("l [m]"); ax[2].set_ylabel("rad/s"); ax[2].set_title("swing frequency of the closed loop", fontsize=8); ax[2].legend(fontsize=7)
    for a_ in ax: a_.grid(True, alpha=0.3)
    fig.tight_layout(); return fig

panel_scale = widgets.param_panel({
    "kp": ("slider", -3.0, 0.0, 0.1, -1.0, "k_phi (kp)"),
    "kd": ("slider", -6.0, 0.0, 0.1, -2.0, "k_phidot (kd)"),
    "m_L": ("slider", 2.0, 6.0, 1.0, 4.0, "payload m_L [kg]"),
    "lag": ("check", True, "motor lag"),
}, draw_scale)

**Step 7: does the linear model describe the nonlinear episode?** The linear response of the 8-state model to a release at angle $\phi_0$ is compared with the nonlinear episode (`QuadLoadPend`, PID+FF+damper from `baselines.py`, 50 Hz ZOH, RK4, saturations active if they occur), same controller gains, hover reference, true-mass feedforward. A small mismatch confirms the derivation in its regime; it grows with $\phi_0$ (A1) and is not a validation outside it.

In [7]:
def linear_vs_nonlinear(kp, kd, phi0, l=2.0, fixed=True, T=6.0, m_L=4.0):
    canon = CANON.with_condition(m_L=m_L, cable_length=l)
    ctrl = canon.controller(m_hat=m_L, damper_gains=None if (kp == 0 and kd == 0) else (kp, kd), fixed_form=fixed)
    r = run_episode(canon.plant(), ctrl, hold_reference((0.0, 5.5)), canon.timing, int(T / 0.02), canon.hover_state(phi=phi0))
    A = closed_loop_A(kp, kd, L=l, m_L=m_L, fixed=fixed, lag=True)
    x0 = np.zeros(8); x0[4] = phi0
    X = np.array([expm(A * tt) @ x0 for tt in r.t])
    return r, X

def draw_validation(preset="damper G(-1,-2)", phi0=0.05, l=2.0, form="fixed"):
    kp, kd = {"arm 1 (no damper)": (0.0, 0.0), "softer G(-1,-0.5)": SOFTER_LAW_GAINS, "damper G(-1,-2)": DAMPER_GAINS}[preset]
    r, X = linear_vs_nonlinear(kp, kd, phi0, l, form == "fixed")
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
    ax[0].plot(r.t, np.degrees(r.phi), **style.arm_style("damper", label="nonlinear episode")); ax[0].plot(r.t, np.degrees(X[:, 4]), **style.arm_style("pid_ff", label="linear model"))
    ax[0].set_title(r"swing $\phi(t)$ [deg]", fontsize=8); ax[0].legend(fontsize=7)
    ax[1].plot(r.t, r.p[:, 0], **style.arm_style("damper", label="nonlinear")); ax[1].plot(r.t, X[:, 0], **style.arm_style("pid_ff", label="linear")); ax[1].set_title("vehicle y [m]", fontsize=8)
    ax[2].plot(r.t, np.degrees(r.phi - X[:, 4]), color="#D55E00"); ax[2].set_title(r"difference $\phi_{NL}-\phi_{lin}$ [deg]", fontsize=8)
    for a_ in ax: a_.set_xlabel("t [s]"); a_.grid(True, alpha=0.3)
    fig.suptitle("max|dphi|/phi0 = %.3f; max|dy| = %.4f m; failed=%s; saturation ticks = %d" % (np.max(np.abs(r.phi - X[:, 4])) / phi0, np.max(np.abs(r.p[:, 0] - X[:, 0])), r.failed, int(r.saturated.sum())), fontsize=8)
    fig.tight_layout(); return fig

panel_val = widgets.param_panel({
    "preset": ("dropdown", ["arm 1 (no damper)", "softer G(-1,-0.5)", "damper G(-1,-2)"], "damper G(-1,-2)", "controller"),
    "phi0": ("slider", 0.02, 0.4, 0.01, 0.05, "release angle phi0 [rad]"),
    "l": ("slider", 1.0, 3.0, 0.25, 2.0, "cable l [m]"),
    "form": ("dropdown", ["fixed", "oracle"], "fixed", "damper form"),
}, draw_validation)

## 5. Interactive exploration

### 5.1 Episode explorer (arms 1 and 2)
Pick a condition $(m_L,l)$ and a scenario; the chosen controllers run on `QuadLoadPend` (50 Hz, RK4 with 10 substeps, allocator with clipping). Standard panels: $y$ and $z$ against the reference, swing $\phi$ with the moving $\phi_{eq}$, tension against the slack line, requested against allocated wrench (orange ticks mark saturation). At $m_L=4$ kg the true-mass and mass-blind PID+FF coincide (H1). Untick *fixed form* to run the oracle form (labelled as such, not as $G$).

In [8]:
def to_record(r):
    return dict(t=r.t, p=r.p, ref_p=r.ref_p, theta=r.theta, phi=r.phi, phi_dot=r.phi_dot, phi_eq=r.phi_eq, T=r.T,
                f_req=r.f_req, tau_req=r.tau_req, f_alloc=r.f_alloc, tau_alloc=r.tau_alloc, sat=r.saturated, failure=r.failure)

@lru_cache(maxsize=256)
def run_arm(L, m_L, arm, scenario, fixed, phi0=None):
    plant, ctrl, ref, ic, n, canon = grid.build_case(CANON, cable_length=L, m_L=m_L, arm=arm, scenario=scenario, fixed_form=fixed, phi0=phi0)
    return run_episode(plant, ctrl, ref, canon.timing, n, ic)

def draw_explorer(m_L=6.0, l=2.0, scenario="hover", ff_true=True, pid_ff=True, softer=False, damper=True, s30match=False, fixed=True):
    style.set_form(fixed)
    arms = [a for a, on in (("ff_true", ff_true), ("pid_ff", pid_ff), ("softer", softer), ("damper", damper), ("s30match", s30match)) if on]
    results = {a: run_arm(l, m_L, a, scenario, fixed) for a in arms}
    runs = {a: to_record(r) for a, r in results.items()}
    ref_rec = next(iter(runs.values()))
    fig = plots.standard_figure(runs, T_nom=m_L * G, ref=ref_rec, title=f"m_L={m_L:g} kg, l={l:g} m, {grid.SCENARIOS[scenario]['label']}")
    msgs = [f"{a}: {r.failure.kind} at {r.failure.time:.2f} s" for a, r in results.items() if r.failed]
    if msgs: fig.text(0.5, 0.005, "FAILURE (episode ended): " + "; ".join(msgs), ha="center", color=style.SLACK_COLOR, fontsize=8)
    return fig

panel_explorer = widgets.param_panel({
    "m_L": ("slider", 2.0, 6.0, 1.0, 6.0, "m_L [kg]"),
    "l": ("slider", 1.0, 3.0, 0.5, 2.0, "cable l [m]"),
    "scenario": ("dropdown", list(grid.SCENARIOS), "hover", "scenario"),
    "ff_true": ("check", True, "true-mass PID+FF (denominator)"),
    "pid_ff": ("check", True, "arm 1 (mass-blind FF)"),
    "softer": ("check", False, "arm 2 softer law (mass-blind FF)"),
    "damper": ("check", True, "damper (mass-blind FF)"),
    "s30match": ("check", False, "softer law, true-mass FF"),
    "fixed": ("check", True, "fixed form G(...) (else oracle form)"),
}, draw_explorer)

### 5.2 Arm 1 in numbers: what the mass-blind feedforward costs
At $L=2$ m, hover recovery and transfer, true-mass against mass-blind feedforward for each payload mass. `z_dev_peak` is the signed peak vehicle height error $z-z_r$ at its largest magnitude; for the mass-blind arm it is negative (a sag) for $m_L>4$ and equals the true-mass value for $m_L<4$ (gate).

In [9]:
rows = []
for scen in ("hover", "transfer"):
    for mL in S30_MASSES:
        rt, rb = run_arm(2.0, float(mL), "ff_true", scen, True), run_arm(2.0, float(mL), "pid_ff", scen, True)
        mt, mb = episode_metrics(rt, mL * G), episode_metrics(rb, mL * G)
        dz = (rb.p[:, 1] - rb.ref_p[:, 1]); dz_t = (rt.p[:, 1] - rt.ref_p[:, 1])
        k = int(np.argmax(np.abs(dz)))
        rows.append(dict(scenario=scen, m_L=mL, pos_rms_true=mt.pos_rms_m, pos_rms_blind=mb.pos_rms_m, c_T_off=c_T_off(mb, mt), c_S_off=c_S_off(mb, mt),
                         z_dev_peak_blind_m=float(dz[k]), z_dev_peak_true_m=float(dz_t[np.argmax(np.abs(dz_t))])))
ARM1 = pd.DataFrame(rows)
display(ARM1.style.format({c: "{:.4f}" for c in ARM1.columns[2:]} | {"m_L": "{:g}"}).hide(axis="index"))

scenario,m_L,pos_rms_true,pos_rms_blind,c_T_off,c_S_off,z_dev_peak_blind_m,z_dev_peak_true_m
hover,2,0.0472,0.0475,1.0079,1.0078,0.0103,-0.0163
hover,3,0.0587,0.0593,1.0102,1.0100,0.0132,-0.0223
hover,4,0.0674,0.0674,1.0000,1.0000,-0.0277,-0.0277
hover,5,0.0742,0.1321,1.7796,0.9554,-0.2180,-0.0328
hover,6,0.0800,0.2314,2.8943,0.9202,-0.4226,-0.0376
transfer,2,0.0529,0.0529,0.9993,1.0146,0.0202,0.0197
transfer,3,0.0669,0.0668,0.9983,1.0162,0.0276,0.0251
transfer,4,0.0777,0.0777,1.0000,1.0000,0.0294,0.0294
transfer,5,0.0865,0.1385,1.6005,0.9307,-0.2056,0.0331
transfer,6,0.0939,0.2300,2.4485,0.8784,-0.4081,0.0362


### 5.3 The offline S30 grid (arms 1 and 2)
**Scope (approved, 2026-11-24):** 25 conditions (cable 1-3 m by payload 2-6 kg) $\times$ 5 controllers $\times$ 2 scenarios = 250 deterministic episodes, one run each, in memory, with the fixed damper form. Metrics: `c_T_off` (vehicle-to-reference position RMS) and `c_S_off` (swing RMS about $\phi_{eq}$, degrees) as ratios to arm 1 with the **true-mass** feedforward at the same condition and scenario; failures (taut exit, value error) are flagged and left blank, never folded into a ratio. These are exploratory and not the canonical `c_T`/`c_S`.

In [10]:
RECOMPUTE = False          # True: rerun the 250 episodes (about 6-25 min with 6 worker processes) and overwrite the CSV
CSV = Path("results/s30_offline_grid.csv")
if RECOMPUTE or not CSV.exists():
    t0 = time.time()
    GRID = grid.run_grid(prov.root, s30_conditions(), workers=6, fixed_form=True)
    grid.save_csv(GRID, CSV, prov, CANON, 6, time.time() - t0)
    GRID_HEADER = ["recomputed in this session"]
else:
    GRID, GRID_HEADER = grid.load_csv(CSV)
print("\n".join(GRID_HEADER))
print("rows: %d; failures: %d" % (len(GRID), int(GRID.failed.sum())))
if GRID.failed.any():
    display(GRID[GRID.failed][["scenario", "L", "m_L", "arm", "failure"]].style.hide(axis="index"))

EXPLORATORY / offline / PROXY: offline S30-style grid, one deterministic run per cell
created: 2026-11-24T14:44:43  workers=6  wall_s=237
thesis_commit: c16a4f8986263d7323cf6299df1b10a5fce8b54e  branch: mrac/w2c-baselines  dirty: False
thesis_root: C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load-mrac-w2c
timing: T_s=0.02 N=10 sim_dt=0.002
vehicle: m=9.6 J=1.5447816 a_p=0.582 T_max=100.518 tau_m=0.05 d=0.3795
gains: PIDGains(kp_y=2.2, kp_z=5.3, kd_y=2.0, kd_z=3.0, kp_theta=25.0, kd_theta=20.0, ki_y=0.6, ki_z=1.8, m=9.6, J=1.5447816, g=9.81, min_thrust=2.0103600000000004, max_thrust=201.036, max_torque=57.91646124, max_pitch_angle=0.6, beta_smoothing=0.0)
beta_tau=0.3 tau_max=57.91646124 L_nom=2.0 m_L_nom=4.0
arms: ff_true, pid_ff, softer, damper, s30match  m_hat_blind=4.0
scenarios: hover: start at hover with the cable released at phi_0 = 0.35 rad (20 deg), reference held at (0, 5.5); transfer: rest-to-rest minimum-jerk 

In [11]:
def draw_grid(scenario="hover", metric="c_S_off"):
    lengths, masses = list(S30_LENGTHS), list(S30_MASSES)
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
    for a_, arm_ in zip(ax, ("pid_ff", "softer", "damper", "s30match")):
        V = grid.to_matrix(GRID, scenario, arm_, metric, lengths, masses)
        failed = ~np.isfinite(V)
        plots.heatmap_grid(a_, lengths, masses, V, f"{ {'pid_ff': 'arm 1 (blind FF)', 'softer': 'arm 2 softer G(-1,-0.5), blind FF', 'damper': 'damper G(-1,-2), blind FF', 's30match': 'softer G(-1,-0.5), true-mass FF'}[arm_] }: {metric}, {grid.SCENARIOS[scenario]['label']}", nominal=(S30_L_NOM, S30_M_NOM),
                           center=1.0, cbar_label="ratio to true-mass PID+FF", failed=failed)
    fig.suptitle("EXPLORATORY / offline / PROXY. All ratios are normalised by the TRUE-MASS PID+FF at the same condition (below 1: better); G(...) = fixed form; square = nominal; x = failed or undefined", fontsize=8)
    fig.tight_layout(); return fig

panel_grid = widgets.param_panel({
    "scenario": ("dropdown", list(grid.SCENARIOS), "hover", "scenario"),
    "metric": ("dropdown", ["c_S_off", "c_T_off", "peak_T_over_Tnom"], "c_S_off", "metric"),
}, draw_grid)

### 5.4 Cross-check with the Pymunk S30 trends (read-only; trends, not values)
**Like-for-like caveat:** the S30 $G(k_p,k_d)$ laws ran with the *true-mass* feedforward and a median-filtered tension gate; in this notebook only the *softer law with true-mass FF* (`s30match`) is like-for-like. The mass-blind softer law and the damper are not. Two things are comparable without any new Pymunk episode: (i) the **swing period of PID+FF against cable length** (S30 validity check V1, `readout.json`, read-only; if the file is not found its numbers from the report are quoted), and (ii) the **ranking of the softer law and the damper**. For (ii) an *exploratory analogue* of the S30 $J_{harm}=2/(c_T+c_S)$ is formed per condition from the offline ratios, with the tracking ratio from the transfer scenario and the swing ratio from hover recovery (mirroring the S30 split of families). It is **not** the S30 $J_{harm}$.

In [12]:
def swing_period(r):
    y = r.phi - 0.0
    idx = np.flatnonzero((y[:-1] > 0) & (y[1:] <= 0))
    tc = r.t[idx] - y[idx] * (r.t[idx + 1] - r.t[idx]) / (y[idx + 1] - y[idx])
    return float(np.mean(np.diff(tc))) if len(tc) >= 2 else math.nan

per = {L: swing_period(run_arm(float(L), 4.0, "ff_true", "hover", True)) for L in S30_LENGTHS}
slope_off = float(np.polyfit(np.log(list(per)), np.log(list(per.values())), 1)[0])
cands = [Path(prov.root) / "experiments/reports/data/S30/sweep/readout.json",
         Path(prov.root).parent / "gym-pymunk-quadrotor-suspended-load/experiments/reports/data/S30/sweep/readout.json"]
readout = next((json.loads(p.read_text()) for p in cands if p.exists()), None)
src = "readout.json (V1, read-only)" if readout else "S30 report (readout.json not found)"
py = readout["V1"]["per_mass"]["m4"]["period_by_L"] if readout else {"1.0": 2.512, "1.5": 2.923, "2.0": 3.27, "2.5": 3.57, "3.0": 3.85}
py_slope = readout["V1"]["per_mass"]["m4"]["slope"] if readout else 0.388
V1 = pd.DataFrame({"L [m]": list(S30_LENGTHS), "offline PID+FF swing period [s] (hover recovery, 4 kg)": [per[L] for L in S30_LENGTHS],
                   "Pymunk S30 V1 period [s] (4 kg)": [py[str(float(L))] for L in S30_LENGTHS],
                   "pendulum 2 pi sqrt(L/g) [s]": [2 * math.pi * math.sqrt(L / G) for L in S30_LENGTHS]})
display(Markdown(f"Swing period of PID+FF against cable length. Source of the Pymunk numbers: {src}. Log-log slope: offline **{slope_off:.3f}**, Pymunk **{py_slope:.3f}**, pendulum 0.5."))
display(V1.style.format("{:.3f}").hide(axis="index"))

piv = GRID.pivot_table(index=["L", "m_L"], columns=["scenario", "arm"], values=["c_T_off", "c_S_off"])
J_off = {}
for arm in ("pid_ff", "softer", "damper", "s30match"):
    J_off[arm] = (2.0 / (piv[("c_T_off", "transfer", arm)] + piv[("c_S_off", "hover", arm)])).dropna()
JT = pd.DataFrame({k: [v.median(), v.min(), v.max(), int(v.size)] for k, v in J_off.items()}, index=["median J_off", "min", "max", "n conditions"]).T
display(Markdown("All ratios are relative to the true-mass PID+FF at the same condition and scenario. Exploratory analogue $J_{harm,off}=2/(c_{T,off}^{transfer}+c_{S,off}^{hover})$ over the 25 conditions (S30 Pymunk medians quoted from the S30 sweep report: softer law 1.296, damper 1.226, 8th of 12 gain pairs; PID+FF with nominal-mass FF is 0.84-0.96 of the true-mass baseline at 5-6 kg)."))
display(JT.style.format({"median J_off": "{:.3f}", "min": "{:.3f}", "max": "{:.3f}", "n conditions": "{:.0f}"}))

Swing period of PID+FF against cable length. Source of the Pymunk numbers: readout.json (V1, read-only). Log-log slope: offline **0.529**, Pymunk **0.388**, pendulum 0.5.

L [m],"offline PID+FF swing period [s] (hover recovery, 4 kg)",Pymunk S30 V1 period [s] (4 kg),pendulum 2 pi sqrt(L/g) [s]
1.000,2.125,2.512,2.006
1.500,2.692,2.923,2.457
2.000,3.121,3.270,2.837
2.500,3.485,3.570,3.172
3.000,3.806,3.850,3.475


All ratios are relative to the true-mass PID+FF at the same condition and scenario. Exploratory analogue $J_{harm,off}=2/(c_{T,off}^{transfer}+c_{S,off}^{hover})$ over the 25 conditions (S30 Pymunk medians quoted from the S30 sweep report: softer law 1.296, damper 1.226, 8th of 12 gain pairs; PID+FF with nominal-mass FF is 0.84-0.96 of the true-mass baseline at 5-6 kg).

,median J_off,min,max,n conditions
pid_ff,0.994,0.333,1.003,25
softer,0.876,0.331,1.048,25
damper,0.743,0.317,0.872,25
s30match,1.012,0.761,1.094,25


## 6. Observations
All numbers below are computed from the executed cells (nothing typed in by hand).

In [13]:
obs = []
# H1
nom = ARM1[(ARM1.m_L == 4)]
h1_identity = float(np.max(np.abs(nom[["c_T_off", "c_S_off"]].values - 1.0)))
off = ARM1[ARM1.m_L > 4]; light = ARM1[ARM1.m_L < 4]
h1_dir = bool((off.c_T_off > 1.0).all()); h1_light = float(np.max(np.abs(light[["c_T_off", "c_S_off"]].values - 1.0)))
h1_gate = float(np.max(np.abs(light[light.scenario == "hover"][["c_T_off", "c_S_off"]].values - 1.0)))
sag6 = ARM1[(ARM1.scenario == "hover") & (ARM1.m_L == 6)].z_dev_peak_blind_m.iloc[0]; sag2 = ARM1[(ARM1.scenario == "hover") & (ARM1.m_L == 2)].z_dev_peak_blind_m.iloc[0]
obs.append(f"**H1.** At 4 kg the two arm-1 variants coincide: max |ratio - 1| = {h1_identity:.1e}. For m_L > 4 c_T_off exceeds 1 in all {len(off)} cases of the table: {h1_dir}; for m_L < 4 the two arms are close but not identical (max |ratio - 1| = {h1_light:.2e}; the gate reads the instantaneous tension, so light-side equality holds at hover only). "
           f"Peak height error (hover recovery, L=2 m): {sag6:+.3f} m at 6 kg (heavier than assumed), {sag2:+.3f} m at 2 kg (lighter).")
# H2/H3
ev0, ev1, ev2 = [zeta_summary(poles(*g_, fixed=True)) for g_ in ((0.0, 0.0), SOFTER_LAW_GAINS, DAMPER_GAINS)]
obs.append(f"**H2 (poles, nominal, motor lag).** least-damped zeta_min: no damper {ev0[0]:.3f}, softer law {ev1[0]:.3f}, damper {ev2[0]:.3f}; max Re(pole) {ev0[1]:.3f}, {ev1[1]:.3f}, {ev2[1]:.3f} (all negative: stable in the linear model).")
errs = {}
for phi0 in (0.05, 0.35):
    r, X = linear_vs_nonlinear(*DAMPER_GAINS, phi0)
    errs[phi0] = (float(np.max(np.abs(r.phi - X[:, 4])) / phi0), float(np.max(np.abs(r.p[:, 0] - X[:, 0]))), int(r.saturated.sum()))
obs.append(f"**H2 (linear vs nonlinear, damper).** max|dphi|/phi0 = {errs[0.05][0]:.3f} at phi0 = 0.05 rad and {errs[0.35][0]:.3f} at 0.35 rad; max |dy| = {errs[0.05][1]:.4f} / {errs[0.35][1]:.4f} m; saturated ticks {errs[0.05][2]} / {errs[0.35][2]}.")
kds = np.linspace(0.0, -30.0, 601); lim = next((k for k in kds if zeta_summary(poles(-1.0, k, fixed=True))[1] > 0), None)
kk = np.linspace(0.0, -10.0, 401)
for tag, mLx in (("nominal m_L = 4 kg", 4.0), ("m_L = 3 kg, m_hat = 4 kg (eps_ff = 0)", 3.0), ("m_L = 6 kg, m_hat = 4 kg (eps_ff = 2/m)", 6.0)):
    eps_x = max(mLx - 4.0, 0.0) / VEH.m
    zz = np.array([zeta_summary(poles(-1.0, k, m_L=mLx, eps_ff=eps_x, fixed=True))[0] for k in kk]); kpk = int(np.argmax(zz))
    zf = lambda kd_: zeta_summary(poles(-1.0, kd_, m_L=mLx, eps_ff=eps_x, fixed=True))[0]
    zsw, _ = track_swing(-1.0, kk, m_L=mLx, eps_ff=eps_x, fixed=True); swf = lambda kd_: float(np.interp(-kd_, -kk, zsw))
    z00 = zeta_summary(poles(0.0, 0.0, m_L=mLx, eps_ff=eps_x, fixed=True))[0]
    lane_a_note = "(Lane A's discrete 50 Hz map at m_L 3, m_hat 4 gives 0.130, 0.200, 0.454, 0.498 for none, softer, damper, kd = -4: four samples, not a monotonicity test; the two models differ by about 0.02 except for the least-damped value at the damper preset, where the sampling delay lowers the attitude pair.)" if mLx == 3.0 else ""
    obs.append(f"**H2, swing-mode damping (tracked by continuation; {tag}).** zeta_swing: (kp = -1, kd = 0) {swf(0.0):.3f}, softer (-0.5) {swf(-0.5):.3f}, damper (-2) {swf(-2.0):.3f}, kd = -4: {swf(-4.0):.3f}; monotone for kd in [0, -4]: {bool(np.all(np.diff(zsw[kk >= -4.0]) >= -1e-9))}, over the whole sweep to -10: {bool(np.all(np.diff(zsw) >= -1e-9))}. {lane_a_note}")
    obs.append(f"**H2, least-damped mode, not monotone ({tag}, L = 2 m, k_phi = -1, motor lag).** zeta_min: true no damper (kp = 0, kd = 0) {z00:.3f}, (kp = -1, kd = 0) {zf(0.0):.3f}, softer (-0.5) {zf(-0.5):.3f}, damper (-2) {zf(-2.0):.3f}, kd = -4: {zf(-4.0):.3f}; maximum {zz[kpk]:.3f} at kd = {kk[kpk]:.2f}; monotone over the sweep: {bool(np.all(np.diff(zz) >= -1e-9))}.")
lim0 = next((k for k in kds if zeta_summary(poles(-1.0, k, fixed=True, d=0.0))[1] > 0), None)
obs.append("**H3.** Linear instability threshold of the rate gain at k_phi = -1 (fixed form, nominal, CONTINUOUS-TIME and UNSATURATED model): " + ("|k_phidot| = %.1f" % -lim if lim is not None else "none up to |k_phidot| = 30") +
           "; the same sweep at d = 0 (no finite zero): " + ("|k_phidot| = %.1f" % -lim0 if lim0 is not None else "no threshold up to 30") +
           ". The selected damper uses |k_phidot| = 2. The 50 Hz sampled-loop threshold was not computed, and the real law clips a_tau at +-1 (outside the linear model).")
Ls = np.linspace(1.0, 3.0, 17)
spread = {f: float(np.ptp([zeta_summary(poles(*DAMPER_GAINS, L=L, fixed=(f == "fixed")))[0] for L in Ls])) for f in ("fixed", "oracle")}
obs.append(f"**H4.** Spread (max-min) of zeta_min over L in [1, 3] m for the damper: fixed form {spread['fixed']:.3f}, oracle form {spread['oracle']:.3f} (hypothesis predicted fixed < oracle: {spread['fixed'] < spread['oracle']}).")
# H5
def frac(metric, scen, arm, cond):
    v = GRID[(GRID.scenario == scen) & (GRID.arm == arm)][metric].dropna(); return int(cond(v).sum()), int(v.size)
for arm in ("softer", "damper", "s30match"):
    a, n1 = frac("c_S_off", "hover", arm, lambda v: v < 1); b, n2 = frac("c_T_off", "hover", arm, lambda v: v > 1)
    a2, n3 = frac("c_S_off", "transfer", arm, lambda v: v < 1); b2, n4 = frac("c_T_off", "transfer", arm, lambda v: v > 1)
    below_arm1 = {s_: int((GRID[(GRID.scenario == s_) & (GRID.arm == arm)].set_index(["L", "m_L"]).c_S_off < GRID[(GRID.scenario == s_) & (GRID.arm == "pid_ff")].set_index(["L", "m_L"]).c_S_off).sum()) for s_ in ("hover", "transfer")}
    obs.append(f"**H5, {arm} (ratios to the true-mass PID+FF).** c_S_off < 1 in {a}/{n1} (hover) and {a2}/{n3} (transfer) conditions, and below arm 1's c_S_off in {below_arm1['hover']} / {below_arm1['transfer']}; c_T_off > 1 in {b}/{n2} (hover) and {b2}/{n4} (transfer).")
obs.append(f"**Failures.** {int(GRID.failed.sum())} of {len(GRID)} episodes ended with a failure record.")
ranks = (J_off["softer"].median() > J_off["damper"].median())
obs.append(f"**Cross-check, ranking.** J_harm,off median softer {J_off['softer'].median():.3f} vs damper {J_off['damper'].median():.3f} (both mass-blind FF): softer above damper = {bool(ranks)}; S30 Pymunk (true-mass FF): softer 1.296 above damper 1.226 = True. Same ordering offline: {bool(ranks)}. Like-for-like arm only: the S30-match (softer, true-mass FF) is {J_off['s30match'].median():.3f} against S30 1.296 (opposite sides of 1: offline it does not beat the denominator on this analogue); the other offline medians use a mass-blind FF and are not comparable. Period slope 0.529 offline, 0.388 Pymunk, 0.5 pendulum: not agreement.")
obs.append(f"**Cross-check.** log-log slope of the PID+FF swing period against L: offline {slope_off:.3f}, Pymunk S30 {py_slope:.3f}, pendulum 0.5. Median J_harm,off (analogue): softer law with true-mass FF (the S30 match) {J_off['s30match'].median():.3f}, softer law with blind FF {J_off['softer'].median():.3f}, damper with blind FF {J_off['damper'].median():.3f}, mass-blind PID+FF {J_off['pid_ff'].median():.3f} (S30 Pymunk: softer law 1.296, damper 1.226, both with true-mass FF).")
display(Markdown("\n\n".join("- " + o for o in obs)))

- **H1.** At 4 kg the two arm-1 variants coincide: max |ratio - 1| = 0.0e+00. For m_L > 4 c_T_off exceeds 1 in all 4 cases of the table: True; for m_L < 4 the two arms are close but not identical (max |ratio - 1| = 1.62e-02; the gate reads the instantaneous tension, so light-side equality holds at hover only). Peak height error (hover recovery, L=2 m): -0.423 m at 6 kg (heavier than assumed), +0.010 m at 2 kg (lighter).

- **H2 (poles, nominal, motor lag).** least-damped zeta_min: no damper 0.158, softer law 0.244, damper 0.378; max Re(pole) -0.317, -0.356, -0.361 (all negative: stable in the linear model).

- **H2 (linear vs nonlinear, damper).** max|dphi|/phi0 = 0.005 at phi0 = 0.05 rad and 0.016 at 0.35 rad; max |dy| = 0.0009 / 0.0110 m; saturated ticks 0 / 0.

- **H2, swing-mode damping (tracked by continuation; nominal m_L = 4 kg).** zeta_swing: (kp = -1, kd = 0) 0.153, softer (-0.5) 0.244, damper (-2) 0.514, kd = -4: 0.535; monotone for kd in [0, -4]: False, over the whole sweep to -10: False. 

- **H2, least-damped mode, not monotone (nominal m_L = 4 kg, L = 2 m, k_phi = -1, motor lag).** zeta_min: true no damper (kp = 0, kd = 0) 0.158, (kp = -1, kd = 0) 0.153, softer (-0.5) 0.244, damper (-2) 0.378, kd = -4: 0.206; maximum 0.419 at kd = -1.70; monotone over the sweep: False.

- **H2, swing-mode damping (tracked by continuation; m_L = 3 kg, m_hat = 4 kg (eps_ff = 0)).** zeta_swing: (kp = -1, kd = 0) 0.115, softer (-0.5) 0.196, damper (-2) 0.436, kd = -4: 0.475; monotone for kd in [0, -4]: False, over the whole sweep to -10: False. (Lane A's discrete 50 Hz map at m_L 3, m_hat 4 gives 0.130, 0.200, 0.454, 0.498 for none, softer, damper, kd = -4: four samples, not a monotonicity test; the two models differ by about 0.02 except for the least-damped value at the damper preset, where the sampling delay lowers the attitude pair.)

- **H2, least-damped mode, not monotone (m_L = 3 kg, m_hat = 4 kg (eps_ff = 0), L = 2 m, k_phi = -1, motor lag).** zeta_min: true no damper (kp = 0, kd = 0) 0.133, (kp = -1, kd = 0) 0.115, softer (-0.5) 0.196, damper (-2) 0.384, kd = -4: 0.210; maximum 0.414 at kd = -1.78; monotone over the sweep: False.

- **H2, swing-mode damping (tracked by continuation; m_L = 6 kg, m_hat = 4 kg (eps_ff = 2/m)).** zeta_swing: (kp = -1, kd = 0) 0.209, softer (-0.5) 0.308, damper (-2) 0.527, kd = -4: 0.531; monotone for kd in [0, -4]: False, over the whole sweep to -10: False. 

- **H2, least-damped mode, not monotone (m_L = 6 kg, m_hat = 4 kg (eps_ff = 2/m), L = 2 m, k_phi = -1, motor lag).** zeta_min: true no damper (kp = 0, kd = 0) 0.263, (kp = -1, kd = 0) 0.209, softer (-0.5) 0.308, damper (-2) 0.316, kd = -4: 0.171; maximum 0.406 at kd = -1.12; monotone over the sweep: False.

- **H3.** Linear instability threshold of the rate gain at k_phi = -1 (fixed form, nominal, CONTINUOUS-TIME and UNSATURATED model): |k_phidot| = 10.0; the same sweep at d = 0 (no finite zero): |k_phidot| = 11.2. The selected damper uses |k_phidot| = 2. The 50 Hz sampled-loop threshold was not computed, and the real law clips a_tau at +-1 (outside the linear model).

- **H4.** Spread (max-min) of zeta_min over L in [1, 3] m for the damper: fixed form 0.112, oracle form 0.166 (hypothesis predicted fixed < oracle: True).

- **H5, softer (ratios to the true-mass PID+FF).** c_S_off < 1 in 25/25 (hover) and 25/25 (transfer) conditions, and below arm 1's c_S_off in 25 / 25; c_T_off > 1 in 25/25 (hover) and 24/25 (transfer).

- **H5, damper (ratios to the true-mass PID+FF).** c_S_off < 1 in 25/25 (hover) and 25/25 (transfer) conditions, and below arm 1's c_S_off in 25 / 25; c_T_off > 1 in 25/25 (hover) and 25/25 (transfer).

- **H5, s30match (ratios to the true-mass PID+FF).** c_S_off < 1 in 25/25 (hover) and 25/25 (transfer) conditions, and below arm 1's c_S_off in 25 / 25; c_T_off > 1 in 25/25 (hover) and 19/25 (transfer).

- **Failures.** 0 of 250 episodes ended with a failure record.

- **Cross-check, ranking.** J_harm,off median softer 0.876 vs damper 0.743 (both mass-blind FF): softer above damper = True; S30 Pymunk (true-mass FF): softer 1.296 above damper 1.226 = True. Same ordering offline: True. Like-for-like arm only: the S30-match (softer, true-mass FF) is 1.012 against S30 1.296 (opposite sides of 1: offline it does not beat the denominator on this analogue); the other offline medians use a mass-blind FF and are not comparable. Period slope 0.529 offline, 0.388 Pymunk, 0.5 pendulum: not agreement.

- **Cross-check.** log-log slope of the PID+FF swing period against L: offline 0.529, Pymunk S30 0.388, pendulum 0.5. Median J_harm,off (analogue): softer law with true-mass FF (the S30 match) 1.012, softer law with blind FF 0.876, damper with blind FF 0.743, mass-blind PID+FF 0.994 (S30 Pymunk: softer law 1.296, damper 1.226, both with true-mass FF).

## 7. Quantitative table

In [14]:
rows = []
for scen in grid.SCENARIOS:
    for arm in ("pid_ff", "softer", "damper", "s30match"):
        sub = GRID[(GRID.scenario == scen) & (GRID.arm == arm)]
        nomrow = sub[(sub.L == S30_L_NOM) & (sub.m_L == S30_M_NOM)].iloc[0]
        rows.append(dict(scenario=scen, arm=arm,
                         c_T_off_median=sub.c_T_off.median(), c_T_off_min=sub.c_T_off.min(), c_T_off_max=sub.c_T_off.max(),
                         c_S_off_median=sub.c_S_off.median(), c_S_off_min=sub.c_S_off.min(), c_S_off_max=sub.c_S_off.max(),
                         c_S_below_1=int((sub.c_S_off < 1).sum()), failures=int(sub.failed.sum()),
                         nominal_c_T_off=nomrow.c_T_off, nominal_c_S_off=nomrow.c_S_off, peak_T_over_Tnom_max=sub.peak_T_over_Tnom.max()))
QT = pd.DataFrame(rows)
display(Markdown("All ratios are relative to the **true-mass PID+FF** at the same condition and scenario (below 1: better than that denominator). arm: pid_ff = arm 1 (blind FF); softer = arm 2 G(-1,-0.5), blind FF; damper = G(-1,-2), blind FF; s30match = G(-1,-0.5) with true-mass FF. G = fixed form."))
display(QT.style.format({c: "{:.3f}" for c in QT.columns if c not in ("scenario", "arm", "c_S_below_1", "failures")}).hide(axis="index"))
display(Markdown("Pole analysis at the nominal condition (Step 4):"))
display(STEP4.style.format({"zeta_min": "{:.3f}", "max_real_part": "{:.3f}"}).hide(axis="index"))

All ratios are relative to the **true-mass PID+FF** at the same condition and scenario (below 1: better than that denominator). arm: pid_ff = arm 1 (blind FF); softer = arm 2 G(-1,-0.5), blind FF; damper = G(-1,-2), blind FF; s30match = G(-1,-0.5) with true-mass FF. G = fixed form.

scenario,arm,c_T_off_median,c_T_off_min,c_T_off_max,c_S_off_median,c_S_off_min,c_S_off_max,c_S_below_1,failures,nominal_c_T_off,nominal_c_S_off,peak_T_over_Tnom_max
hover,pid_ff,1.012,1.000,5.919,1.000,0.911,1.012,10,0,1.000,1.000,1.100
hover,softer,2.039,1.478,6.145,0.830,0.761,0.935,25,0,1.507,0.844,1.125
hover,damper,4.398,3.077,8.808,0.756,0.663,0.851,25,0,3.619,0.765,1.128
hover,s30match,1.645,1.327,2.606,0.844,0.777,0.934,25,0,1.507,0.844,1.109
transfer,pid_ff,1.006,0.982,5.043,1.000,0.870,1.021,10,0,1.000,1.000,1.176
transfer,softer,1.487,0.997,5.116,0.649,0.583,0.997,25,0,1.065,0.639,1.117
transfer,damper,1.987,1.529,5.455,0.487,0.432,0.652,25,0,1.529,0.494,1.073
transfer,s30match,1.106,0.935,1.851,0.669,0.591,0.995,25,0,1.065,0.639,1.098


Pole analysis at the nominal condition (Step 4):

controller,motor_lag,zeta_min,max_real_part,oscillatory_modes
arm 1 (no damper),True,0.158,-0.317,"-1.02+0.79j, -0.32+1.98j, -7.20+16.44j"
arm 1 (no damper),False,0.170,-0.338,"-1.02+0.77j, -0.34+1.96j"
"softer law G(-1,-0.5)",True,0.244,-0.356,"-0.56+0.86j, -0.60+2.41j, -7.21+16.10j"
"softer law G(-1,-0.5)",False,0.265,-0.356,"-0.57+0.86j, -0.66+2.39j"
"damper G(-1,-2)",True,0.378,-0.361,"-0.44+1.07j, -1.68+2.81j, -6.96+14.95j"
"damper G(-1,-2)",False,0.375,-0.361,"-0.43+1.08j, -1.87+2.94j"


## 8. Interpretation (EXPLORATORY)

* **Damping is not monotone in the rate gain.** In the continuous-time linear model of this notebook, $\zeta_{\min}$ of the least-damped mode rises from the arm-1 value to a maximum at a moderate $k_{\dot\phi}$ and then falls, because the identity of the least-damped mode changes along the sweep. **Mode identity depends on the model:** here the least-damped mode at the damper preset is the pair near 1.1 rad/s; lane A's discrete 50 Hz map gives the 15 rad/s attitude-loop pair instead ($\zeta$ 0.331 there against 0.378 here; the sampling delay lowers that pair's damping). Neither is wrong. The tracked *swing-mode* $\zeta$ is higher at $k_{\dot\phi}=-4$ than at $0$ but is **not monotone on the dense sweep** (the observation cell prints `False` for $[0,-4]$ and for the whole sweep); lane A's 'monotone' statement rests on four samples ($0,-0.5,-2,-4$) and is not a monotonicity test. All thresholds and $\zeta$ values are for the **continuous-time, unsaturated** model: the 50 Hz sampled-loop threshold was not computed, and the real law clips $a_\tau$ at $\pm1$, which the linear threshold ignores. 'More gain' is not a safe direction.
* **What the derivation gives.** With the vertical loop decoupled and no saturation, the damper enters only the attitude equation and only through $\Phi$, so the closed-loop polynomial is affine in $(K_\phi,K_{\dot\phi})$ with a gain-independent numerator $N(s)=g_e s^3(d\,m\,s^2-g(m+m_L))$. The mechanism is the cascade of the brief: torque $\to$ tilt (through the PID attitude loop) $\to$ lateral force $\to$ motion of the attachment point $\to$ swing. The offset attachment ($d>0$) gives the damper path a real zero pair $\pm\sqrt{g(m+m_L)/(dm)}$, one in the right half plane; the cause is the $d\,\ddot\theta$ term of the swing equation (the attachment point first moves opposite to the tilt-driven motion), **not** the cable torque $dT\sigma$. The zero guarantees only that infinite gain is unstable; the finite limit seen in the sweep is a fast pair reaching the imaginary axis (H3, numbers above, including the $d=0$ comparison).
* **Regime.** All of this is linear, continuous-time and unsaturated (A1-A8). The nonlinear check agrees for small release angles and the mismatch stays small even at 0.35 rad (about 1.6 % of the amplitude, section 6); it is a check of the derivation in its regime, not a stability result. It is *not* a stability result for the sampled, saturated, Pymunk system.
* **Arm 1.** The mass-blind feedforward costs nothing at 4 kg and for lighter payloads (the tension gate removes the surplus), and sags for heavier payloads; the ratios in section 7 quantify the tracking cost in this offline model only.
* **Arm 2.** The softer law and the damper trade tracking for swing in the offline grid in the directions listed in the observations; whether the trend resembles Pymunk S30 is judged only by the cross-check numbers (period slope, ranking of $J_{harm,off}$), not by values; only the softer law with true-mass FF is like-for-like with S30, and its offline $J_{harm,off}$ and the period slope do **not** agree with the Pymunk numbers.
* **Limits of the grid.** Two scenarios, one run each, deterministic; 25 conditions; the offline plant is a taut point mass (Pymunk: an 8-segment chain with different slack behaviour); no wind, no kicks; the metric definitions differ from the canonical ones. The nominal condition of the transfer and hover scenarios was fixed before the roll; no scenario or gain was tuned after seeing a result.

## 9. Decisions and next steps

* **Next (notebook 07):** arm 3 (payload-aware MRAC law in the A0 architecture), formulation, $Q$-gate and a PRELIMINARY run against these arms.
* **For Javier:** (i) whether the damper's right-half-plane zero (for $d>0$) and the gain limit seen in the linear sweep should be flagged in the thesis plan for the cooperative extension, where $d$ may differ per vehicle; (ii) whether the oracle or the fixed form is the arm-2 comparator for arm 3 (this notebook uses the fixed form; the explorer toggles both); (iii) the independent check of the derivation of section 4.3 (planned: an Opus review before the meeting) is **not** done by this notebook.
* **NOT RUN here:** Pymunk episodes, any training, any claim about stability of the real closed loop.